# 04 · Databricks：训练集和参考数据变成表（bronze → silver）

## 这一章在整个项目里的位置

```
00–03b 进货 ✅ → [04 训练集和参考数据：拆箱、清洗] → 05 dbt → 06 自动化 → 07–09 训练模型、预测全蛋白组 → 10 对外 API
```

## 背景：为什么先做训练集

我们的**最终目标**：用新的流程重新训练 QAFI，再给全部人类蛋白重新预测一遍。现在手上那份全蛋白组预测是旧方法算的，不一定对。

顺序就按目标来：**先把训练集整理干净 → 训练模型 → 预测全蛋白组。** 所以这一章的主线是：
- **训练集**：ProteinGym 的 DMS 实验数据，就是实验测出来的"某个突变让蛋白功能变好还是变坏"
- **参考数据**：UniProt（核对蛋白编号）、ClinVar（以后评估模型用的临床标签）

旧的全蛋白组预测也准备好了，放在附录里：它在 09 号用来做新旧对比。

## 你在哪里操作

```
你：在这个 notebook 里运行一格（一行命令）
        ↓
Databricks 命令行工具（你装好、登录好的那个）把代码和任务送上去
        ↓
Databricks 在云上干活：搬文件、建表、清洗、检查
        ↓
你：在这里看结果，或者到 Databricks 网页上看（第 8 步告诉你点哪里）
```

**网页是用来看的，不是用来手动建东西的。** Databricks 上的一切也写成代码、放进 Git，和 01 号的 Terraform 是一个道理。

## 这一章做的 8 件事

| # | 做什么 | 用的技术 | 比喻 |
|---|---|---|---|
| 1 | 建好数据的"目录" | Unity Catalog | 一个环境一栋楼，按加工程度分层 |
| 2 | 把 Databricks 上的东西写成图纸并部署 | Asset Bundles | 实验方案 |
| 3 | 把文件从 AWS 搬进 Databricks | 临时下载链接 | 一次性提货单 |
| 4 | 运行任务：几个步骤自动按顺序跑 | Lakeflow Jobs | 照方案做实验 |
| 5 | bronze：原样变成表 | Delta 表 | 拆箱，原样上架 |
| 6 | silver：一行代表什么，清洗 | Spark SQL | 清点、去重、贴对标签 |
| 7 | 数据质量检查 | 检查规则 | 出厂质检 |
| 8 | 去网页上看结果 | Catalog、血缘、任务记录 | 参观仓库 |

**花费：0**。Databricks 免费版有用量上限，这一章远在额度之内。

> 不认识的词（Unity Catalog、Delta、Spark、Job），先看[基础知识](../../docs/primer_aws_databricks.md)第 3 节。

---

## 准备

运行下面这一格：定义执行命令的小工具，连上 AWS 和 Databricks。**不用看代码。**

In [ ]:
import os, subprocess
from pathlib import Path
import boto3
from databricks.sdk import WorkspaceClient
from qafi_platform import lakehouse
from qafi_platform.lakehouse import sql
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ["AWS_PROFILE"] = "qafi"
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
aws = boto3.Session(region_name="eu-central-1")
s3, w = aws.client("s3"), WorkspaceClient(profile="qafi")
CATALOG, JOB, RAW = "qafi_dev", "qafi training data", "qafi-186972156153-dev-raw"
print("AWS:       ", aws.client("sts").get_caller_identity()["Arn"])
print("Databricks:", w.config.host)

**结果怎么看**：一行 AWS 身份（`pipi-admin`），一行 Databricks 工作区地址。

> - AWS 报"登录过期"：在终端运行 `aws login --profile qafi`，再运行这一格
> - Databricks 报错：在终端运行 `databricks auth login --profile qafi`，再运行这一格

---

## 第 1 步 · Unity Catalog：数据的目录和门禁

### 在干什么
在 Databricks 里建好放数据的"目录"。

### 背景
数据一多，就要回答三个问题：**东西放在哪？谁能看？是从哪来的？** 散在各处的文件回答不了。

### 用了什么技术，为什么
**Unity Catalog** 是 Databricks 的数据目录和门禁系统。比喻：一个环境一栋楼，楼里按"加工程度"分层。

| 层 | 比喻 | 我们的 |
|---|---|---|
| **catalog**（目录） | 一栋楼：一个环境一栋 | `qafi_dev`（开发环境） |
| **schema**（分区） | 楼里的一层：按加工程度 | `raw`（原料）、`bronze`（原样的表）、`silver`（清洗过的表） |
| **table**（表） | 书架上的一套书 | 比如 `qafi_dev.silver.dms_scores` |
| **volume**（文件区） | 资料室：放不是表的文件 | `qafi_dev.raw.files`，放从 AWS 搬来的文件 |

任何一张表都用三段名字找到：`哪栋楼.哪一层.哪张表`。Unity Catalog 还管**谁能读哪张表**，并**自动记录血缘**：每张表是从哪张表、哪个文件来的（第 8 步在网页上看）。

**为什么叫 `qafi_dev`**：公司里每个环境一栋楼（`_dev` 开发、`_prod` 正式），两栋楼里表名一样、数据互不干扰。dsm 课程时你建过一个 `qafi`，那是练习用的，不动它。

### 执行
楼只建一次。公司里由平台团队用 Terraform 建；我们一行 SQL：

In [ ]:
sql(w, f"CREATE CATALOG IF NOT EXISTS {CATALOG} COMMENT 'QAFI ML platform, development environment'")
sql(w, "SHOW CATALOGS")

### 结果怎么看
列表里有 `qafi_dev`。楼里的三层和文件区不在这里建，在第 2 步由图纸建。

> 第一次运行 SQL 可能要等十几秒：SQL 计算资源（warehouse）平时是关着的，用的时候自己启动，不用不收钱。

### 面试怎么说
> "Data is organised in Unity Catalog as catalog.schema.table, with one catalog per environment and schemas by layer; it governs access and records lineage automatically."

---

## 第 2 步 · Asset Bundles：Databricks 上的东西也写成图纸

### 在干什么
用图纸把楼层、文件区和任务部署到 Databricks。

### 背景
dsm 课程时，你是在网页上手动建 Job、手动上传文件的，就像做实验不写方案、全凭记忆：这次做出来了，下次再做、或者换个人做，很可能哪一步不一样。

### 用了什么技术，为什么
**Databricks Asset Bundles** 就是写下来的**实验方案**，和你总结的 Terraform 一样："记录了怎么建，以后照着图纸造就行"。图纸在 [`databricks/`](../../databricks/) 文件夹里：

| 文件 | 比喻 | 管什么 | 多久变一次 |
|---|---|---|---|
| [`databricks.yml`](../../databricks/databricks.yml) | 方案的封面 | 项目叫什么、在哪个工作区做 | 几乎不变 |
| [`resources/lakehouse.yml`](../../databricks/resources/lakehouse.yml) | 方案的流程图 | 三层楼、文件区、两个任务，每个任务分几步、谁等谁 | 加步骤时变 |
| [`src/notebooks/`](../../databricks/src/notebooks/) | 每一步的详细操作 | 每一步具体怎么做 | 最常变 |
| [`src/lakehouse/`](../../databricks/src/lakehouse/) | 几步共用的标准操作 | 在本地有测试，CI 每次都验证 | 偶尔变 |

**改细节，只改那一个 `.py`；加步骤、删步骤、调顺序，才改流程图。**

图纸里有两个任务（后面的章节又加了三个：07d 的 `qafi structures`、07e 的 `qafi language model`、07f 的 `qafi baseline model`）：
- `qafi training data`：训练集和参考数据，**这一章跑的就是它**
- `qafi old proteome predictions`：旧的全蛋白组预测，09 号再跑（见附录）

三个命令：

| 命令 | 比喻 |
|---|---|
| `databricks bundle validate` | 先检查方案有没有写错 |
| `databricks bundle deploy` | 照方案把实验室布置好：楼层建好，操作说明放到位 |
| `databricks bundle run` | 开始做实验 |

### 公司里什么情况怎么做
- **改了某一步的代码**：再 `deploy` 一次，新代码就上去了。公司里由 CI 在合并后自动 deploy 到正式环境
- **不要在网页上手动加步骤**：图纸里没有的东西，下次 `deploy` 会被覆盖掉。可以先在网页的 notebook 里试，确定了再写进图纸

### 执行

In [ ]:
run("cd databricks && databricks bundle validate")
run("cd databricks && databricks bundle deploy")

### 结果怎么看
- validate：`Validation OK!`
- deploy：没有变化时显示 `unchanged`；改过代码会显示上传了几个文件

### 面试怎么说
> "Everything on Databricks is defined in an Asset Bundle in git — schemas, volumes, jobs and their code — and deployed with the CLI, the same way Terraform manages the AWS side."

---

## 第 3 步 · 把文件从 AWS 搬进 Databricks

### 在干什么
让 Databricks 拿到 raw 里的五个文件：ProteinGym 的两个、UniProt 的两个、ClinVar 一个。

### 背景
Databricks 免费版**不能直接接我们自己的 S3 桶**。付费版可以：Unity Catalog 用"存储凭证 + 外部位置"直接读 S3，数据不用复制，原理和 02 号让 GitHub 扮演 AWS 角色一样。

### 用了什么技术，为什么
**临时下载链接**（presigned URL）：在你电脑上，用你的 AWS 身份给每个文件签一个链接。它**只能读**，**6 小时后自动失效**，**不需要把任何 AWS 密码交给 Databricks**。比喻：一次性提货单。

链接清单（加上每个文件应有的大小）放进 Databricks 的文件区；任务的第一步按清单下载，下载完核对大小。已经在的、大小对的文件会跳过，所以重跑没关系。

**路径和 S3 里一模一样**（`datasets/<数据集>/<版本>/<文件>`）。以后换成付费版，只要把文件区建在 S3 上，代码不用改，只是不再需要"搬"这一步。

### 执行

In [ ]:
KEYS = [
    "datasets/proteingym/v1.3/DMS_ProteinGym_substitutions.zip",
    "datasets/proteingym/v1.3/DMS_substitutions.csv",
    "datasets/uniprot-human/2026_03/UP000005640_9606.fasta.gz",
    "datasets/uniprot-human/2026_03/UP000005640_9606.idmapping.gz",
    "datasets/clinvar/2026-10/variant_summary_2026-10.txt.gz",
]
lakehouse.upload_transfer_list(w, CATALOG, lakehouse.transfer_links(s3, RAW, KEYS, hours=6))

### 结果怎么看
显示清单的位置：`/Volumes/qafi_dev/raw/files/_transfer/links.json`。

### 面试怎么说
> "On Free Edition the workspace can't mount our bucket, so files travel through short-lived presigned URLs — read-only, six hours, no AWS credentials in Databricks — into a managed volume with the same paths as S3. With a paid workspace the volume would sit on an external location and the copy step would disappear."

---

## 第 4 步 · 运行任务

### 在干什么
启动 `qafi training data` 任务，五个步骤自动按顺序跑：

```
copy_raw ──┬──▶ bronze_reference ──┬──▶ silver_dms ──▶ quality_checks
           └──▶ bronze_dms ─────────┘
```

| 步骤 | 做什么 |
|---|---|
| `copy_raw` | 按清单把文件下载进文件区 |
| `bronze_reference` | ClinVar、UniProt → bronze 表 |
| `bronze_dms` | ProteinGym → bronze 表；和上一步**同时跑** |
| `silver_dms` | 两个 bronze 都好了才开始：清洗成训练集（第 6 步讲） |
| `quality_checks` | 检查训练集，不合格就让整个任务失败（第 7 步讲） |

### 背景
几个步骤手动一个个跑：要人盯着、容易漏、出错不知道停。**Job** 把它们串成流水线：谁先谁后、谁和谁同时跑，都写在图纸里。

### 用了什么技术，为什么
- **Lakeflow Jobs**：Databricks 的任务调度。06 号会让它定时自动跑
- **Serverless**：不用自己开机器、关机器，Databricks 自动分配，跑完就释放

### 执行
启动后这一格马上结束，任务在 Databricks 上跑，大约 5 分钟：

In [ ]:
run("cd databricks && databricks bundle run training_data --no-wait")

看进度。隔一两分钟再运行一次，就能看到最新状态：

In [ ]:
lakehouse.latest_run(w, JOB)

### 结果怎么看
- 第一行是任务在网页上的地址，点开能看到五个方框一个个变绿（07b/07c 往这个任务里加了 MaveDB 两步、06 号加了 `gold`，所以你现在看到的是八个方框）
- 表格里每个步骤一行：`RUNNING` 在跑，`BLOCKED` 在等前面的步骤，`TERMINATED` 加 `SUCCESS` 是完成了
- 全部 `SUCCESS` 再往下做

### 公司里什么情况怎么做
- **某一步失败了**：点进那一步看错误信息，修好代码，`deploy`，然后在网页上点 **Repair run**：**只重跑失败的那一步和它后面的**，前面成功的不用重来。我写这一章时就用过（见附录）

### 面试怎么说
> "The pipeline is a Lakeflow Job on serverless compute with its dependencies declared in the bundle; a failed run is repaired from the failed task instead of starting over."

---

## 第 5 步 · bronze：原样变成表

### 在干什么
看 `bronze_dms` 和 `bronze_reference` 装进来了什么。

### 背景
bronze 的规矩是"**原样**"：一个字不改。这样任何时候都能对照原始文件；以后发现清洗规则错了，也能从 bronze 重新来，不用再去拆箱。

### 用了什么技术，为什么
**ProteinGym 是什么样的**：217 个 DMS 实验，每个实验一个文件，每行是一个突变（比如 `A16C`）和它测出来的分数；另外一张**实验说明表**，写着每个实验测的是哪个蛋白、什么物种、蛋白序列。物种分布：人类 96 个实验，原核生物 50 个，其他真核生物 40 个，病毒 31 个。

**每一行都记着出处**：`_source_file`（来自哪个文件）、`_dataset_version`（哪个版本）、`_ingested_at`（什么时候装的）。DMS 的每一行还记着它属于哪个实验文件（`DMS_filename`）。

**ClinVar 的列名里有空格和括号**，比如 `RS# (dbSNP)`。Delta 表默认不允许这种列名，我第一次试跑就卡在这里。为了原样保留，这几张表打开了 Delta 的 **column mapping**（列名映射）功能；查这种列时要用反引号括起来：`` `RS# (dbSNP)` ``。

**Delta 表**：Databricks 的表格式，底下是 Parquet 文件，外加一本"修改日志"：
- 每次写入要么全部成功，要么全部不算，不会写一半
- **能回到过去任何一个版本**（time travel）：误删了、写错了，都能找回来

### 执行
bronze 里的五张表各有多少行：

In [ ]:
sql(w, f'''
SELECT 'proteingym_dms' AS bronze_table, count(*) AS rows FROM {CATALOG}.bronze.proteingym_dms
UNION ALL SELECT 'proteingym_assays', count(*) FROM {CATALOG}.bronze.proteingym_assays
UNION ALL SELECT 'clinvar_variant_summary', count(*) FROM {CATALOG}.bronze.clinvar_variant_summary
UNION ALL SELECT 'uniprot_proteome', count(*) FROM {CATALOG}.bronze.uniprot_proteome
UNION ALL SELECT 'uniprot_idmapping', count(*) FROM {CATALOG}.bronze.uniprot_idmapping''')

Delta 表的修改日志：

In [ ]:
sql(w, f"DESCRIBE HISTORY {CATALOG}.bronze.proteingym_dms")[["version", "timestamp", "operation"]]

### 结果怎么看
- `proteingym_dms` 约 247 万行：单点突变和多点突变都在
- `proteingym_assays` 217 行：一个实验一行
- `clinvar_variant_summary` 约 920 万行，`uniprot_proteome` 20,652 个蛋白，`uniprot_idmapping` 约 446 万行
- 修改日志里每次写入一行。以后要看某个旧版本：`SELECT * FROM 表 VERSION AS OF 0`

### 面试怎么说
> "Bronze keeps each source exactly as delivered, with lineage columns on every row; Delta gives atomic writes and time travel, and column mapping keeps source column names that contain spaces."

---

## 第 6 步 · silver：训练集，一行代表什么

### 在干什么
把 DMS 数据清洗成训练集：`silver.dms_scores`。

### 背景
写这一章时，我先看了数据：
1. **72% 的行是多点突变**（比如 `A1C:D2E`，好几个突变的组合）。QAFI 预测的是**单点突变**的效果
2. 突变写成一串文字（`A16C`），要拆成位置 16、原来的氨基酸 A、换成的氨基酸 C，才能和其他数据连起来
3. 实验说明表里有蛋白序列，可以拿来**核对**：突变写的"原来的氨基酸"，必须和序列在那个位置上的一样，否则说明位置编号错位了
4. 蛋白用的是 UniProt 的 entry name（比如 `P53_HUMAN`），要换成 UniProt 编号（`P04637`），才能和全蛋白组那边连起来

### 用了什么技术，为什么
数据工程里最核心的一个问题：**这张表，一行代表什么？**（叫 grain，粒度）。

`silver.dms_scores` 的答案：**一行 = 一个实验里的一个单点突变**。
- 同一个突变如果被好几个实验测过（比如 TP53 有好几个 DMS 实验），就是好几行，每个实验一行
- **只留单点突变**；多点突变留在 bronze 里，不丢
- 突变拆成 `position`、`ref_aa`、`alt_aa`
- `ref_matches_sequence`：原来的氨基酸是否和实验的蛋白序列对得上。按规矩必须全部对上，第 7 步检查
- **分数原样保留**：ProteinGym 已经把方向统一成"越高 = 功能越好"。每个实验内部要不要缩放到 0–1，是**建模的选择**，放到 07 号
- `uniprot_id`：从 UniProt 参考蛋白组里，用 entry name 查到的编号（人类蛋白）

`silver.dms_assay_audit`：每个实验一行，记录"多少行进来，多少单点突变留下，多少多点突变留在 bronze"。

### 执行
bronze 和 silver 各有多少行，按物种分：

In [ ]:
sql(w, f"SELECT taxon, count(*) AS assays, sum(bronze_rows) AS bronze_rows, sum(single_substitutions) AS kept_in_silver, sum(multi_mutants_left_in_bronze) AS multi_left_in_bronze FROM {CATALOG}.silver.dms_assay_audit GROUP BY taxon ORDER BY assays DESC")

TP53 的 R175H（癌症里最常见的突变之一），被哪几个实验测过、分数是多少：

In [ ]:
sql(w, f"SELECT dms_id, uniprot_id, mutant, dms_score, dms_score_bin, selection_type FROM {CATALOG}.silver.dms_scores WHERE uniprot_entry_name = 'P53_HUMAN' AND position = 175 AND alt_aa = 'H'")

人类实验里，有没有蛋白没对上 UniProt 编号：

In [ ]:
sql(w, f"SELECT DISTINCT dms_id, uniprot_entry_name FROM {CATALOG}.silver.dms_scores WHERE taxon = 'Human' AND uniprot_id IS NULL")

### 结果怎么看
- 人类 96 个实验，32.9 万个单点突变进了 silver，对应 80 个人类蛋白；所有物种加起来是 69.6 万个，另外 177 万个多点突变留在 bronze
- TP53 R175H 被 4 个实验测过（Giacomelli 2018 的三种条件、Kotler 2018），各一行，`uniprot_id` 是 `P04637`。分数都是负的，也就是功能变差，和它是已知的癌症热点突变一致
- 只有一个人类蛋白没对上：`Q53Z42_HUMAN`，它不在 UniProt 的参考蛋白组里。不硬猜，留着空值，以后用别的对照表补

### 公司里什么情况怎么做
- **训练数据和预测数据一样要清洗**：机器学习那一层只拿干净的表，不自己拆原始文件
- **每张表都写明 grain**：表的说明里就写着"一行代表什么"，网页上能看到（第 8 步）

### 面试怎么说
> "The training table has an explicit grain — one row per single substitution per assay. Multi-mutants stay in bronze, each mutant's reference residue is checked against the assay sequence, and proteins are mapped to UniProt accessions so they join with the proteome-wide data."

---

## 第 7 步 · 数据质量检查

### 在干什么
训练集必须满足的规则，一条条检查。

### 背景
清洗代码写错了、上游数据变了，结果表照样能生成，只是数据是错的，而且没人发现。训练集错了，后面训练出来的模型也是错的。质量检查是**出厂质检**：不合格就停下。

### 用了什么技术，为什么
每条规则是一句 SQL，满足就返回"是"：

| 规则 | 类型 |
|---|---|
| 每个有单点突变的实验都进了 silver | 硬 |
| 一个实验里，一个突变只有一行 | 硬 |
| 原来的氨基酸和实验的蛋白序列对得上 | 硬 |
| 氨基酸是标准的 20 种之一，而且确实变了 | 硬 |
| 每个突变都有分数 | 硬 |
| 至少 95% 的人类实验对上了 UniProt 编号 | 软 |

**硬规则**不满足，整个任务失败；**软规则**不满足，只记一个警告。每次的结果都追加到 `silver.quality_checks` 表里，能查到历史。

### 执行
最近一次检查的结果：

In [ ]:
sql(w, f"SELECT scope, check, passed, hard FROM {CATALOG}.silver.quality_checks WHERE run_at = (SELECT max(run_at) FROM {CATALOG}.silver.quality_checks)")

### 结果怎么看
每条规则一行，`passed` 应该都是 `true`。"原来的氨基酸和序列对得上"这一条，70 万行一个都没错：说明 ProteinGym 的位置编号可靠，我们的拆分也写对了。

### 公司里什么情况怎么做
- **检查失败**：任务停下，发报警。先查是代码错了还是数据变了，修好再重跑
- **05 号**：用 dbt 把这类规则写成 dbt 的测试，和数据转换写在一起

### 面试怎么说
> "Quality checks gate the pipeline: hard rules such as one row per substitution per assay, or reference residues matching the assay sequence, fail the job before anyone trains on the data; every result is kept in a table."

---

## 第 8 步 · 去 Databricks 网页上看

打开你的工作区：准备那一格打印出来的地址。

| 去哪 | 看什么 |
|---|---|
| 左边 **Catalog** → `qafi_dev` → `silver` → `dms_scores` | **Overview**：每一列的类型和说明；**Sample Data**：前几行数据 |
| 同一张表 → **Lineage** 标签 | **血缘图**：`dms_scores` 是从 `bronze.proteingym_dms`、`bronze.proteingym_assays`、`bronze.uniprot_proteome` 来的。这些不用写代码，自动记录 |
| `bronze` → `proteingym_dms` → **History** | 第 5 步看到的修改日志 |
| 左边 **Jobs & Pipelines** → `qafi training data` | 每次运行的记录：五个步骤、各花了多久、有没有出错 |
| 左边 **SQL Editor** | 自己写 SQL，比如 `SELECT * FROM qafi_dev.silver.dms_scores WHERE uniprot_entry_name = 'BRCA1_HUMAN' LIMIT 100` |

---

## 提交代码

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)   # 第一次会自动整理 notebook 的格式
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 04 training data and references on Databricks") && git push -q')

---

## 附录 · 旧的全蛋白组预测（09 号再跑）

### 它是什么，为什么现在不跑
这是旧方法算出来的全蛋白组预测（2 万多个蛋白，约 5.7 亿行）。新模型训练好以后，09 号会给全蛋白组重新预测，再拿这份旧结果**做新旧对比**。现在先不跑：它要 40–60 分钟，眼下用不上。

处理它的代码已经写好、用 20 个蛋白试通了，是图纸里的第二个任务 `qafi old proteome predictions`：

```
copy_raw ──▶ bronze_proteome ──▶ silver_proteome ──▶ quality_checks
```

### 写它的时候已经发现的问题
1. **大量完全重复的行**：第一个蛋白 Q16720 有 43% 的行一模一样
2. **交叉连接**：同一个变异还有几百上千行（Q16720 最多 729 行，P48551 最多 19,683 行）。当年把基因组位置、REVEL、BayesDel 两两乱配了一遍；QAFIMeta 是用它们算的，所以也跟着变。其他分数在每份拷贝里都一样（试跑 0 个冲突）
3. **不是每个文件的列都一样**：有的蛋白文件少了 qafi1、QAFImt、qafisplit1–3 这些列，因为不同的 QAFI 模型覆盖的蛋白不一样。少的列记成空值，并写明少了哪些
4. **`GeneID` 列里放的其实是 entry name**（比如 `P53_HUMAN`），silver 里改名为 `entry_name`，真正的基因编号从 UniProt 查
5. **我的假设错了，质检当场拦住了**：我以为 QAFI 分数都是 0 到 1 的概率。试跑时质检失败：qafi1 最大 1.45（QAFI 是回归模型，预测的是 DMS 分数），EVE 那一列是原始分数（−12 到 17）。改好规则后，用 **Repair run** 只重跑了质检这一步

它的 silver（`silver.variant_scores`）是**一个变异一行**，只放没被污染的蛋白层面分数；被交叉连接污染的列不放进去，因为判断不了哪个组合是真的，硬猜就是造数据。

### 到时候怎么跑
09 号会带你跑：链接清单里加上全蛋白组压缩包，然后 `databricks bundle run old_predictions`。

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 目录 | Unity Catalog | 数据放在哪、谁能看、从哪来 |
| 2 图纸 | Asset Bundles | Databricks 上的东西也能审查、能重建 |
| 3 搬运 | 临时下载链接 | 不把 AWS 密码交给别的平台 |
| 4 流水线 | Lakeflow Jobs + serverless | 步骤自动按顺序跑，失败只重跑失败的部分 |
| 5 bronze | Delta 表 | 原样保存，能追溯，能回到旧版本 |
| 6 silver | 粒度（grain） | 一行代表什么说清楚，确定不了的不硬猜 |
| 7 质检 | 检查规则 | 坏数据进不了训练 |

**一句话**：训练集第一次变成了一张干净的表，一个实验里的一个单点突变一行，每一行都核对过；参考数据也都进了湖仓。下一步就可以在它上面做特征、训练模型。

## 面试题

1. **What is the medallion architecture?**
   > "Layers of increasing quality: bronze keeps the source as delivered, silver is cleaned with a defined grain and checks, gold holds tables built for a use case. Each layer can be rebuilt from the one before."

2. **What does 'grain' mean, and why does it matter?**
   > "What one row represents. For the DMS training data it's one single substitution per assay, so a variant measured by three assays is three rows. In the legacy proteome file, defining the grain exposed a join fan-out of up to 19,683 rows per variant."

3. **How do you validate data you parse?**
   > "Against the source itself: every DMS mutant's reference residue is checked against the assay's protein sequence. Zero mismatches in 700,000 rows confirms both the publisher's numbering and my parsing."

4. **How do you handle schema drift between source files?**
   > "Load missing columns as NULL and record which ones were missing, so an absent column is distinguishable from an empty cell; an unknown column stops the load for a human to look at."

**下一章 05 · dbt**：在 silver 的基础上，用 dbt 写出最终给训练和 API 用的 gold 表，并给每张表加上测试和文档。